# LNA gain measurement with ADALM-PLUTO

まずPlutoへ接続し、現在の設定を読み出して疎通を確認します。  
このプロジェクトでは2チャネルを扱うため、`adi.Pluto`ではなく`adi.ad9361`を使用します。


## 1. Plutoへ接続

USB経由のネットワークURI `ip:192.168.2.1` を使用します。  
このセルではRF設定を変更せず、現在値の読み出しだけを行います。

windows dll_relative_pathを指定する必要があるので，このような面倒なコードになってる．


In [ ]:
import os
from pathlib import Path
from importlib.util import find_spec

# Windowsではpylibiioが使用するDLLの場所を、adiのimport前に指定する
if os.name == "nt": #これはwindowsのときだけ実行される
    dll_relative_path = Path("tmp/libiio-v0.26/Windows-VS-2022-x64")
    search_roots = [Path.cwd(), *Path.cwd().parents]
    libiio_dir = next(
        (root / dll_relative_path for root in search_roots if (root / dll_relative_path).is_dir()),
        None,
    )
    if libiio_dir is None:
        raise FileNotFoundError(
            "libiio.dllのフォルダが見つかりません。Notebookをこのプロジェクト内で実行してください。"
        )
    libiio_dir = libiio_dir.resolve()
    os.environ["PATH"] = str(libiio_dir) + os.pathsep + os.environ.get("PATH", "")
    # Python 3.8以降のWindows DLL検索にも登録し、ハンドルを保持する
    if "libiio_dll_handle" not in globals():
        libiio_dll_handle = os.add_dll_directory(str(libiio_dir))

if find_spec("adi") is None: # adiがインストールされていない場合は、エラーを出す
    raise ModuleNotFoundError(
        "pyadi-iioがありません。直前のインストールセルを実行し、"
        "カーネルを再起動してください。"
    )

import adi

PLUTO_URI = "ip:192.168.2.1"

# このセルを再実行した場合は、古い接続を先に閉じる
if "sdr" in globals() and sdr is not None:  #type: ignore
    try:
        sdr.close() #type: ignore
    except Exception:
        pass

try:
    sdr = adi.ad9361(uri=PLUTO_URI) #type: ignore
except Exception as exc:
    sdr = None
    raise ConnectionError(
        f"Plutoへ接続できませんでした ({PLUTO_URI})。"
        "USB接続、Plutoの起動、192.168.2.1への到達性を確認してください。"
    ) from exc

# 属性を実機から読み返せれば接続成功
pluto_status = {
    "URI": PLUTO_URI,
    "RX LO [Hz]": int(sdr.rx_lo),
    "TX LO [Hz]": int(sdr.tx_lo),
    "Sample rate [S/s]": int(sdr.sample_rate),
    "RX RF bandwidth [Hz]": int(sdr.rx_rf_bandwidth),
}

print("Pluto connection: OK")
for name, value in pluto_status.items():
    print(f"  {name}: {value}")


Pluto connection: OK
  URI: ip:192.168.2.1
  RX LO [Hz]: 2400000000
  TX LO [Hz]: 2400000000
  Sample rate [S/s]: 30720000
  RX RF bandwidth [Hz]: 20000000


In [3]:
import numpy as np
import matplotlib.pyplot as plt

sdr.tx_enabled_channels = [0, 1]

sdr.tx_lo = int(2.4e9)
sdr.tx_sample_rate = int(2e6)
sdr.tx_rf_bandwidth = int(1e6)
sdr.tx_hardwaregain_chan0 = -70
sdr.tx_hardwaregain_chan1 = -70


In [ ]:
N = 65536
f_bb = 100e3
phase_deg = 90

n = np.arange(N)
x = 0.5 * 2**14 * np.exp(
    1j * 2 * np.pi * f_bb * n / sdr.tx_sample_rate
)

tx1 = x
tx2 = x * np.exp(1j * np.deg2rad(phase_deg))